## Step 0 — Download a Public Dataset (Optional)

If you don't have your own images, run this cell to download a small public
dataset straight into `/content/real_images/` — which is exactly what
`IMAGE_DIR` points to.

**Available choices** (set `DATASET_CHOICE` below):

| `DATASET_CHOICE` | Images | Subject |
|---|---|---|
| `"nelorth/oxford-flowers"` | 8,189 | Flowers 🌸 |
| `"pcuenq/oxford-pets"` | 7,349 | Cats & Dogs 🐾 |
| `"Multimodal-Fatima/Smithsonian_Butterflies_subset"` | 1,000 | Butterflies 🦋 |

Run **only one** dataset cell, then continue from Step 1.
> Skip this cell entirely if you are uploading your own images.

In [ ]:
import os
from pathlib import Path
from PIL import Image
from datasets import load_dataset

# ── Pick your dataset ────────────────────────────────────────────────────────
DATASET_CHOICE = "nelorth/oxford-flowers"   # change to any option from the table above
MAX_IMAGES     = 500                         # set to None to download all images
# ────────────────────────────────────────────────────────────────────────────

IMAGE_DIR = "/content/real_images"          # must match the IMAGE_DIR in Step 2
os.makedirs(IMAGE_DIR, exist_ok=True)

print(f"Downloading '{DATASET_CHOICE}' …")
ds = load_dataset(DATASET_CHOICE, split="train")

if MAX_IMAGES is not None:
    ds = ds.select(range(min(MAX_IMAGES, len(ds))))

print(f"Saving {len(ds)} images to {IMAGE_DIR} …")
saved = 0
for i, sample in enumerate(ds):
    # HuggingFace image datasets expose a PIL Image under 'image' or 'img'
    img = sample.get("image") or sample.get("img")
    if img is None:
        continue  # skip samples with no image column

    img = img.convert("RGB")
    dest = Path(IMAGE_DIR) / f"img_{i:05d}.jpg"
    img.save(dest, format="JPEG", quality=95)
    saved += 1
    if saved % 100 == 0 or saved == len(ds):
        print(f"  Saved {saved}/{len(ds)}")

print(f"\n✓ {saved} images ready in {IMAGE_DIR}")
print("  Continue to Step 1 (Install Dependencies) →")

# Fine-Tuning SD-Turbo with LoRA

This notebook fine-tunes [Stable Diffusion Turbo (SD-Turbo)](https://huggingface.co/stabilityai/sd-turbo)
on your own images using **LoRA** (Low-Rank Adaptation).

LoRA is a parameter-efficient training technique: instead of updating all ~860 M UNet weights,
it injects small trainable rank-decomposition matrices into selected attention layers.
This keeps VRAM usage low and training fast — ideal for Colab.

**Pipeline overview:**
1. Install dependencies
2. Set your configuration
3. Load & preprocess images
4. Attach LoRA adapters to the UNet
5. Run the DDPM fine-tuning loop
6. Save the adapter weights

## Step 1 — Install Dependencies

Run the cell below to install all required libraries.
This only needs to be done once per Colab session (or when setting up a fresh environment).

In [ ]:
!pip install diffusers peft accelerate transformers torch Pillow

## Step 2 — Configuration

Edit the variables in the cell below before running the notebook:

| Variable | Description |
|---|---|
| `IMAGE_DIR` | Folder containing your training images (`.jpg` / `.png`) |
| `OUTPUT_DIR` | Where the trained LoRA adapter weights will be saved |
| `NUM_EPOCHS` | How many full passes over your dataset to train for |
| `LEARNING_RATE` | Step size for the AdamW optimiser — `1e-4` is a safe default |
| `LORA_RANK` | Rank of LoRA matrices; higher = more capacity, more VRAM |

In [ ]:
# ── User-editable configuration ──────────────────────────────────────────────
IMAGE_DIR     = "/content/real_images"   # folder of training images
OUTPUT_DIR    = "/content/lora_output"   # where LoRA weights are saved
NUM_EPOCHS    = 50                        # training epochs
LEARNING_RATE = 1e-4                      # AdamW learning rate
LORA_RANK     = 4                         # rank of LoRA decomposition matrices
# ─────────────────────────────────────────────────────────────────────────────

print("Configuration:")
print(f"  IMAGE_DIR     = {IMAGE_DIR}")
print(f"  OUTPUT_DIR    = {OUTPUT_DIR}")
print(f"  NUM_EPOCHS    = {NUM_EPOCHS}")
print(f"  LEARNING_RATE = {LEARNING_RATE}")
print(f"  LORA_RANK     = {LORA_RANK}")

## Step 3 — Dataset Class

The `RealImageDataset` below does three things for every image in `IMAGE_DIR`:

1. **Load** — uses Pillow to open the file and convert it to RGB.
2. **Resize** — scales to `256×256` pixels so all tensors are the same shape.
3. **Normalize** — converts pixel values from `[0, 255]` to `[-1, 1]`, which is
   the range SD-Turbo was trained on (mean `0.5`, std `0.5` per channel).

The `DataLoader` will batch these tensors together for efficient GPU training.

In [ ]:
import os
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

# Supported image extensions.
_IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

# Transform: resize → tensor → normalize to [-1, 1]
_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.ToTensor(),                              # [0, 255] → [0.0, 1.0]
    transforms.Normalize(mean=[0.5, 0.5, 0.5],          # [0.0, 1.0] → [-1.0, 1.0]
                         std=[0.5, 0.5, 0.5]),
])


class RealImageDataset(Dataset):
    """PyTorch Dataset that loads every image in a directory."""

    def __init__(self, image_dir: str):
        self.paths = sorted(
            p for p in Path(image_dir).rglob("*")
            if p.is_file() and p.suffix.lower() in _IMAGE_EXTS
        )
        if len(self.paths) == 0:
            raise ValueError(f"No images found in {image_dir!r}")
        print(f"Found {len(self.paths)} image(s) in {image_dir}")

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> torch.Tensor:
        img = Image.open(self.paths[idx]).convert("RGB")
        return _transform(img)


# Quick sanity check — will run when IMAGE_DIR exists.
if os.path.isdir(IMAGE_DIR):
    dataset = RealImageDataset(IMAGE_DIR)
    sample = dataset[0]
    print(f"Sample tensor shape : {sample.shape}")   # expected: torch.Size([3, 256, 256])
    print(f"Pixel value range   : [{sample.min():.2f}, {sample.max():.2f}]")
else:
    print(f"Skipping dataset check — {IMAGE_DIR!r} not found yet.")

## Step 4 — LoRA Setup

**What is LoRA?**
Instead of training all UNet weights, LoRA adds a pair of low-rank matrices
`A` (rank × d_in) and `B` (d_out × rank) to selected weight matrices.
Only `A` and `B` are updated during training — everything else is frozen.
For rank=4 this reduces trainable parameters by ~99 % vs full fine-tuning.

**Why these attention layers?**
`to_q`, `to_k`, `to_v` are the query/key/value projections in each attention
block — they control what the model "pays attention to" and carry most of the
domain-specific visual style. `to_out.0` is the output projection that
recombines the attention heads.

In [ ]:
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
from peft import LoraConfig, get_peft_model

MODEL_ID = "stabilityai/sd-turbo"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ── Load the noise scheduler ───────────────────────────────────────────────
scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

# ── Load the UNet (the part we will fine-tune) ────────────────────────────
print("Loading UNet from", MODEL_ID, "...")
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet")
unet = unet.to(device)

# ── Load the VAE (frozen — used only to encode images to latent space) ─────
print("Loading VAE ...")
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae")
vae = vae.to(device)
vae.requires_grad_(False)   # freeze VAE weights

# ── Attach LoRA adapters to UNet attention layers ──────────────────────────
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,          # alpha == rank keeps the effective LR stable
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.0,
    bias="none",
)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

## Step 5 — Training Loop

This cell implements the standard **DDPM denoising objective**:

$$\mathcal{L} = \mathbb{E}_{x_0,\, \epsilon \sim \mathcal{N}(0,I),\, t}\bigl[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\bigr]$$

At each training step:
1. Encode the image batch into the VAE latent space (`z`).
2. Sample a random diffusion timestep `t` and a noise tensor `ε`.
3. Mix the latent and noise according to the scheduler to produce a noisy latent `z_t`.
4. Ask the UNet to predict the noise that was added (`ε_pred`).
5. Compute MSE loss between `ε_pred` and the true `ε` and back-propagate.

Because we only fine-tune the LoRA parameters, memory requirements are low
and training converges quickly even with a small dataset.

In [ ]:
import torch.nn.functional as F

# ── DataLoader ────────────────────────────────────────────────────────────
dataset    = RealImageDataset(IMAGE_DIR)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=2)

# ── Optimiser (only LoRA params require grad) ─────────────────────────────
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=LEARNING_RATE,
)

# ── SD-Turbo uses a fixed empty-string conditioning — we use a zero tensor ─
# The unconditional cross-attention context expected by the SD-Turbo UNet
# has shape (batch, seq_len=77, channels=1024).
ENCODER_HIDDEN_DIM = 1024
SEQ_LEN            = 77

unet.train()
global_step = 0

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss = 0.0

    for step, pixel_values in enumerate(dataloader):
        pixel_values = pixel_values.to(device)          # (B, 3, 256, 256)

        # 1. Encode images to VAE latent space (scaling factor = 0.18215)
        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * vae.config.scaling_factor   # (B, 4, 32, 32)

        # 2. Sample noise and a random timestep for each image in the batch
        noise      = torch.randn_like(latents)
        batch_size = latents.shape[0]
        timesteps  = torch.randint(
            0, scheduler.config.num_train_timesteps,
            (batch_size,), device=device,
        ).long()

        # 3. Forward diffusion: mix latent with noise at timestep t
        noisy_latents = scheduler.add_noise(latents, noise, timesteps)

        # 4. UNet predicts the noise (unconditional: zero encoder hidden states)
        encoder_hidden_states = torch.zeros(
            batch_size, SEQ_LEN, ENCODER_HIDDEN_DIM, device=device
        )
        noise_pred = unet(
            noisy_latents,
            timesteps,
            encoder_hidden_states=encoder_hidden_states,
        ).sample

        # 5. DDPM loss: MSE between predicted and actual noise
        loss = F.mse_loss(noise_pred, noise)

        # 6. Back-propagate and update LoRA weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss  += loss.item()
        global_step += 1

        if global_step % 10 == 0:
            avg = epoch_loss / (step + 1)
            print(f"  Epoch {epoch:>3}/{NUM_EPOCHS} | step {global_step:>5} | "
                  f"loss {loss.item():.6f} | avg {avg:.6f}")

    print(f"Epoch {epoch:>3}/{NUM_EPOCHS} complete — "
          f"avg loss: {epoch_loss / len(dataloader):.6f}")

print("\nTraining complete.")

## Step 6 — Save LoRA Weights

`unet.save_pretrained()` writes a small `adapter_model.bin` (or `adapter_model.safetensors`)
alongside an `adapter_config.json` to `OUTPUT_DIR`.  Only the LoRA matrices are saved —
**not** the full UNet — so the file is typically just a few MB.

To reload later in `pipeline/generator.py`:
```python
from diffusers import UNet2DConditionModel
from peft import PeftModel

base_unet = UNet2DConditionModel.from_pretrained("stabilityai/sd-turbo", subfolder="unet")
unet      = PeftModel.from_pretrained(base_unet, OUTPUT_DIR)
```

In [ ]:
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save only the LoRA adapter weights (base UNet weights are NOT saved).
unet.save_pretrained(OUTPUT_DIR)

print(f"LoRA weights saved to {OUTPUT_DIR}")
print(f"\nFiles written:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:40s}  {size_kb:8.1f} KB")